In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

In [ ]:
dataset = datasets.ImageFolder(
    root="dataset",
    transform=transform
)

classes = dataset.classes
print(classes)

In [ ]:
train_loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True
)

In [ ]:
class HybridModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.convnext = convnext_tiny(
            weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1
        )
        self.convnext.classifier = nn.Identity()

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.vit = timm.create_model(
            "vit_base_patch16_224",
            pretrained=True
        )
        self.vit.head = nn.Identity()

        self.gru = nn.GRU(
            input_size=1536,
            hidden_size=256,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(512,8)   # ← 8 classes

    def forward(self,x):

        conv_feat = self.convnext.features(x)
        conv_feat = self.pool(conv_feat).flatten(1)

        vit_feat = self.vit(x)

        features = torch.cat([conv_feat,vit_feat],dim=1)

        features = features.unsqueeze(1)

        out,_ = self.gru(features)

        out = out[:,-1,:]

        out = self.fc(out)

        return out

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = HybridModel().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [ ]:
epochs = 10

for epoch in range(epochs):

    model.train()

    for images,labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("Epoch completed:",epoch)

In [ ]:
torch.save(model.state_dict(),"model_8class.pth")

In [ ]:
from PIL import Image

classes = dataset.classes

img = Image.open("test.jpg").convert("RGB")

img = transform(img).unsqueeze(0)

model.eval()

with torch.no_grad():

    output = model(img)

    pred = torch.argmax(output,1).item()

print("Prediction:",classes[pred])